In [3]:
import duckdb
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
DB_PATH      = PROJECT_ROOT / "nhs_ae.duckdb"

con = duckdb.connect(str(DB_PATH))

report_df = con.execute("""
    SELECT
        period,
        org_code,
        org_name,
        parent_org,
        type1_attendances,
        type2_attendances,
        other_attendances,
        total_attendances,
        over_4h_type1,
        within_4h_type1,
        four_hour_pct_type1,
        wait_12plus_dta
    FROM reporting.ae_monthly
    ORDER BY period, org_code
""").df()

print(f"Report dataset: {len(report_df)} rows, {report_df['period'].nunique()} months, "
      f"{report_df['org_code'].nunique()} providers")

Report dataset: 987 rows, 5 months, 199 providers


In [4]:
monthly = con.execute("""
    SELECT
        period,
        COUNT(*)                                AS providers_reporting,
        SUM(type1_attendances)                  AS type1_attendances,
        SUM(other_attendances)                  AS other_attendances,
        SUM(total_attendances)                  AS total_attendances,
        SUM(within_4h_type1)                    AS within_4h_type1,
        SUM(over_4h_type1)                      AS over_4h_type1,
        ROUND(
            100.0 * SUM(within_4h_type1)
            / NULLIF(SUM(type1_attendances), 0)
        , 1)                                    AS national_4h_pct_type1,
        SUM(wait_12plus_dta)                    AS total_12plus_waits
    FROM reporting.ae_monthly
    GROUP BY period
    ORDER BY period
""").df()

print("=== National monthly summary ===\n")
print(monthly.to_string(index=False))

=== National monthly summary ===

    period  providers_reporting  type1_attendances  other_attendances  total_attendances  within_4h_type1  over_4h_type1  national_4h_pct_type1  total_12plus_waits
2025-11-01                  197          1420033.0           794353.0          2263562.0         853403.0       566630.0                   60.1             50648.0
2025-12-01                  197          1407641.0           785243.0          2239924.0         835326.0       572315.0                   59.3             50789.0
2026-01-01                  197          1405035.0           780500.0          2235696.0         801402.0       603633.0                   57.0             71517.0
2026-02-01                  197          1271830.0           723986.0          2043572.0         751846.0       519984.0                   59.1             54649.0
2026-03-01                  199          1451010.0           845475.0          2351347.0         927024.0       523986.0                   63.9   

In [5]:
provider_report = con.execute("""
    SELECT
        period,
        org_code,
        org_name,
        type1_attendances,
        within_4h_type1,
        four_hour_pct_type1,
        wait_12plus_dta
    FROM reporting.ae_monthly
    WHERE type1_attendances > 0
    ORDER BY period, four_hour_pct_type1 ASC
""").df()

# Show the 10 lowest performing providers in the most recent month
latest = report_df["period"].max()
print(f"=== 10 lowest Type 1 performers — {latest} ===\n")
bottom10 = (provider_report[provider_report["period"] == latest]
            .head(10))
print(bottom10.to_string(index=False))

=== 10 lowest Type 1 performers — 2026-03-01 00:00:00 ===

    period org_code                                                 org_name  type1_attendances  within_4h_type1  four_hour_pct_type1  wait_12plus_dta
2026-03-01      RK9                  UNIVERSITY HOSPITALS PLYMOUTH NHS TRUST               9053             3390                 37.4              352
2026-03-01      RX1                NOTTINGHAM UNIVERSITY HOSPITALS NHS TRUST              13228             5906                 44.6             1200
2026-03-01      RXW            THE SHREWSBURY AND TELFORD HOSPITAL NHS TRUST              11446             5165                 45.1             1173
2026-03-01      RJE         UNIVERSITY HOSPITALS OF NORTH MIDLANDS NHS TRUST              15324             7136                 46.6              820
2026-03-01      RBT              MID CHESHIRE HOSPITALS NHS FOUNDATION TRUST               7509             3563                 47.4              665
2026-03-01      RN3             GRE

In [6]:
# Step 5 verification requirement:
# Recalculate two published figures directly from the cleaned table
# and confirm they match what the report shows

print("=== Verification: recalculated vs report figures ===\n")

# Figure 1: Total Type 1 attendances for March 2026
calc_total = con.execute("""
    SELECT SUM(type1_attendances)
    FROM reporting.ae_monthly
    WHERE period = '2026-03-01'
""").fetchone()[0]

report_total = monthly[monthly["period"] == pd.Timestamp("2026-03-01")]["type1_attendances"].values[0]

print(f"March 2026 Type 1 attendances:")
print(f"  Summed from provider rows : {calc_total:,.0f}")
print(f"  National summary row      : {report_total:,.0f}")
print(f"  Match: {'✅ YES' if calc_total == report_total else '❌ NO'}\n")

# Figure 2: National 4-hour performance for November 2025
calc_pct = con.execute("""
    SELECT ROUND(
        100.0 * SUM(within_4h_type1) / NULLIF(SUM(type1_attendances), 0)
    , 1)
    FROM reporting.ae_monthly
    WHERE period = '2025-11-01'
""").fetchone()[0]

report_pct = monthly[monthly["period"] == pd.Timestamp("2025-11-01")]["national_4h_pct_type1"].values[0]

print(f"November 2025 national 4-hour performance:")
print(f"  Recalculated from components : {calc_pct}%")
print(f"  National summary row         : {report_pct}%")
print(f"  Match: {'✅ YES' if calc_pct == report_pct else '❌ NO'}")

=== Verification: recalculated vs report figures ===

March 2026 Type 1 attendances:
  Summed from provider rows : 1,451,010
  National summary row      : 1,451,010
  Match: ✅ YES

November 2025 national 4-hour performance:
  Recalculated from components : 60.1%
  National summary row         : 60.1%
  Match: ✅ YES


In [7]:
output_path = PROJECT_ROOT / "docs" / "ae_provider_report.csv"
provider_report.to_csv(output_path, index=False)
print(f"Report exported: {output_path}")
print(f"Rows: {len(provider_report)}")

Report exported: C:\Users\q8x7u\Documents\cv\CV\NHS project\nhs-ae-pipeline\docs\ae_provider_report.csv
Rows: 605


In [8]:
con.close()
print("Connection closed")

Connection closed
